# Sleeper API Weekly Stats - Automated Update

Automatically fetches the latest week's NFL stats from Sleeper's public API.

**Schedule:** Runs weekly on Tuesdays at 8:00 AM (America/Chicago)

**Features:**
- ✅ 100% free, no API key required
- ✅ No authentication needed
- ✅ No rate limits
- ✅ Auto-detects current NFL season and week
- ✅ Only fetches new data we don't already have
- ✅ 2,000+ players with detailed stats

**Source:** Sleeper API (https://api.sleeper.app/v1)

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime

# Sleeper API configuration
BASE_URL = "https://api.sleeper.app/v1"

print("✓ Sleeper API configured")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Historical backfill for all weeks in 2024 and 2025
# Run this ONCE to populate historical data

import time

print("="*70)
print("SLEEPER API - HISTORICAL BACKFILL")
print("="*70)
print("\nThis will fetch all available weeks for 2024 and 2025.")
print("⚠️  This is a one-time backfill - use the 'Fetch Latest Week Data' cell for ongoing updates\n")

# Fetch player metadata once for all seasons (reuse for efficiency)
print("Fetching Sleeper player metadata (one-time)...")
try:
    players_response = requests.get(f"{BASE_URL}/players/nfl", timeout=30)
    sleeper_players = players_response.json() if players_response.status_code == 200 else {}
    print(f"✓ Loaded {len(sleeper_players)} player records\n")
except Exception as e:
    print(f"❌ Error fetching player metadata: {e}")
    sleeper_players = {}

# Define seasons and weeks to backfill
backfill_config = [
    {"season": 2024, "weeks": range(1, 19)},  # Regular season weeks 1-18
    {"season": 2025, "weeks": range(1, 19)}   # Regular season weeks 1-18
]

total_records = 0
successful_weeks = []
failed_weeks = []

for config in backfill_config:
    season = config["season"]
    weeks = config["weeks"]
    
    print(f"\n{'='*70}")
    print(f"Processing Season {season}")
    print(f"{'='*70}")
    
    for week in weeks:
        try:
            print(f"\n[Week {week}] Fetching data...")
            
            # Fetch weekly stats from Sleeper
            season_type = 'regular'
            
            response = requests.get(
                f"{BASE_URL}/stats/nfl/{season_type}/{season}/{week}",
                timeout=30
            )
            
            if response.status_code != 200:
                print(f"  ⚠️  Stats unavailable (status {response.status_code}) - skipping")
                failed_weeks.append((season, week, f"Status {response.status_code}"))
                time.sleep(0.5)
                continue
            
            weekly_stats = response.json()
            
            if not weekly_stats or len(weekly_stats) == 0:
                print(f"  ⚠️  No stats found - skipping")
                failed_weeks.append((season, week, "No stats"))
                continue
            
            print(f"  Found {len(weekly_stats)} players")
            
            # Build rows combining player info and weekly stats
            rows = []
            for player_id, stats in weekly_stats.items():
                # Get player metadata
                player_info = sleeper_players.get(player_id, {})
                player_name = player_info.get('full_name', player_info.get('first_name', '') + ' ' + player_info.get('last_name', '')).strip() or 'Unknown'
                position = player_info.get('position', 'Unknown')
                team = player_info.get('team', 'FA')
                
                # Calculate fantasy points from stats (PPR scoring)
                fantasy_pts = float(stats.get('pts_ppr', 0) or 0)
                
                # Combine player info and stats
                combined_stats = {
                    'player_name': player_name,
                    'position': position,
                    'team': team,
                    'stats': stats,
                    'player_metadata': player_info
                }
                
                rows.append(
                    Row(
                        player_id=player_id,
                        week=int(week),
                        season=int(season),
                        fantasy_points=fantasy_pts,
                        stats=json.dumps(combined_stats),
                        source='sleeper'
                    )
                )
            
            if not rows:
                print(f"  ⚠️  No valid player data - skipping")
                failed_weeks.append((season, week, "No valid data"))
                continue
            
            stats_df = spark.createDataFrame(rows)
            
            # Write to bronze
            bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
            bronze_df.createOrReplaceTempView("sleeper_backfill_bronze")
            
            spark.sql("""
                MERGE INTO main.fantasai.bronze_weekly_stats AS target
                USING sleeper_backfill_bronze AS source
                ON target.player_id = source.player_id 
                    AND target.week = source.week 
                    AND target.season = source.season
                    AND target.source = source.source
                WHEN MATCHED THEN
                    UPDATE SET
                        target.fantasy_points = source.fantasy_points,
                        target.stats = source.stats,
                        target.ingested_at = source.ingested_at
                WHEN NOT MATCHED THEN
                    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
            """)
            
            # Write to silver
            silver_df = bronze_df.dropDuplicates(["player_id", "week", "season", "source"])
            silver_df.createOrReplaceTempView("sleeper_backfill_silver")
            
            spark.sql("""
                MERGE INTO main.fantasai.silver_weekly_stats AS target
                USING sleeper_backfill_silver AS source
                ON target.player_id = source.player_id 
                    AND target.week = source.week 
                    AND target.season = source.season
                    AND target.source = source.source
                WHEN MATCHED THEN
                    UPDATE SET
                        target.fantasy_points = source.fantasy_points,
                        target.stats = source.stats,
                        target.ingested_at = source.ingested_at
                WHEN NOT MATCHED THEN
                    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
            """)
            
            player_count = bronze_df.count()
            total_records += player_count
            successful_weeks.append((season, week))
            
            print(f"  ✓ Stored {player_count} unique players")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            failed_weeks.append((season, week, str(e)))
            continue
        
        # Rate limiting between weeks (be respectful)
        time.sleep(0.5)

print("\n" + "="*70)
print("BACKFILL COMPLETE")
print("="*70)
print(f"\n✓ Successfully processed {len(successful_weeks)} weeks")
print(f"✓ Total unique players stored: {total_records}")

if failed_weeks:
    print(f"\n⚠️  Failed weeks ({len(failed_weeks)}):")
    for season, week, reason in failed_weeks[:10]:
        print(f"  - {season} Week {week}: {reason}")
    if len(failed_weeks) > 10:
        print(f"  ... and {len(failed_weeks) - 10} more")

print("\n💡 Next: Use the 'Fetch Latest Week Data' cell for ongoing updates")

In [0]:
# Auto-detect current season and find latest week we DON'T have yet
print("="*70)
print("SLEEPER API - LATEST DATA FETCH")
print("="*70)
print(f"\nCurrent date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Determine current season
current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

if current_month >= 9:  # September onwards = current year's season
    CURRENT_SEASON = current_year
else:  # January-August = previous year's season
    CURRENT_SEASON = current_year - 1

print(f"Detected NFL season: {CURRENT_SEASON}")

# Check what weeks we already have in the database for this season
print("\nChecking existing data in database...")
existing_weeks = spark.sql(f"""
    SELECT DISTINCT week 
    FROM main.fantasai.silver_weekly_stats 
    WHERE season = {CURRENT_SEASON} AND source = 'sleeper'
    ORDER BY week
""").collect()

existing_week_numbers = [row['week'] for row in existing_weeks]

if existing_week_numbers:
    print(f"Already have Sleeper data for weeks: {existing_week_numbers}")
    WEEK = max(existing_week_numbers) + 1 if existing_week_numbers else 1
else:
    print("No Sleeper data found for this season yet")
    WEEK = 1

# Don't go past week 18 (regular season)
if WEEK > 18:
    print(f"\n⚠️  Week {WEEK} exceeds regular season (weeks 1-18)")
    print("All weeks are already up to date!")
    dbutils.notebook.exit("No new data to fetch - all weeks current")

print(f"\nWill fetch: Season {CURRENT_SEASON}, Week {WEEK}")

try:
    # Fetch weekly stats from Sleeper
    print(f"\nFetching Sleeper stats for Week {WEEK}...")
    
    # Sleeper stats endpoint: /stats/nfl/{season_type}/{season}/{week}
    season_type = 'regular'
    
    response = requests.get(
        f"{BASE_URL}/stats/nfl/{season_type}/{CURRENT_SEASON}/{WEEK}",
        timeout=30
    )
    response.raise_for_status()
    weekly_stats = response.json()
    
    if not weekly_stats or len(weekly_stats) == 0:
        print(f"\n⚠️  No data available yet for Week {WEEK}")
        print("Games may not have been played or data not yet published")
        dbutils.notebook.exit(f"No data available for Season {CURRENT_SEASON} Week {WEEK}")
    
    print(f"✓ Fetched stats for {len(weekly_stats)} players\n")
    
    # Fetch player metadata once (for player names, positions, teams)
    print("Fetching player metadata...")
    players_response = requests.get(f"{BASE_URL}/players/nfl", timeout=30)
    sleeper_players = players_response.json() if players_response.status_code == 200 else {}
    print(f"✓ Loaded {len(sleeper_players)} player records\n")
    
    # Build rows combining player info and weekly stats
    rows = []
    for player_id, stats in weekly_stats.items():
        # Get player metadata
        player_info = sleeper_players.get(player_id, {})
        player_name = player_info.get('full_name', player_info.get('first_name', '') + ' ' + player_info.get('last_name', '')).strip() or 'Unknown'
        position = player_info.get('position', 'Unknown')
        team = player_info.get('team', 'FA')
        
        # Calculate fantasy points from stats (PPR scoring)
        fantasy_pts = float(stats.get('pts_ppr', 0) or 0)
        
        # Combine player info and stats
        combined_stats = {
            'player_name': player_name,
            'position': position,
            'team': team,
            'stats': stats,
            'player_metadata': player_info
        }
        
        rows.append(
            Row(
                player_id=player_id,
                week=int(WEEK),
                season=int(CURRENT_SEASON),
                fantasy_points=fantasy_pts,
                stats=json.dumps(combined_stats),
                source='sleeper'
            )
        )
    
    stats_df = spark.createDataFrame(rows)
    
    # Write to bronze
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    bronze_df.createOrReplaceTempView("sleeper_bronze_temp")
    
    spark.sql("""
        MERGE INTO main.fantasai.bronze_weekly_stats AS target
        USING sleeper_bronze_temp AS source
        ON target.player_id = source.player_id 
            AND target.week = source.week 
            AND target.season = source.season
            AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET
                target.fantasy_points = source.fantasy_points,
                target.stats = source.stats,
                target.ingested_at = source.ingested_at
        WHEN NOT MATCHED THEN
            INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
            VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    # Write to silver
    silver_df = bronze_df.dropDuplicates(["player_id", "week", "season", "source"])
    silver_df.createOrReplaceTempView("sleeper_silver_temp")
    
    spark.sql("""
        MERGE INTO main.fantasai.silver_weekly_stats AS target
        USING sleeper_silver_temp AS source
        ON target.player_id = source.player_id 
            AND target.week = source.week 
            AND target.season = source.season
            AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET
                target.fantasy_points = source.fantasy_points,
                target.stats = source.stats,
                target.ingested_at = source.ingested_at
        WHEN NOT MATCHED THEN
            INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
            VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    player_count = bronze_df.count()
    
    print("="*70)
    print("FETCH COMPLETE")
    print("="*70)
    print(f"\n✓ Season: {CURRENT_SEASON}")
    print(f"✓ Week: {WEEK}")
    print(f"✓ Players stored: {player_count}")
    print(f"\n📅 Data ingested at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
    raise